<a href="https://colab.research.google.com/github/ananyeahh06-ops/react-agent-assignment-2/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [13]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [14]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam

In [15]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {

    "pizza_facts": (
        "Pizza originated in Naples, Italy, in the 18th century. "
        "The classic Margherita pizza uses tomato, mozzarella, and basil. "
        "Neapolitan pizza is baked in a wood-fired oven at very high heat. "
        "Pizza became popular worldwide after Italian immigrants spread it abroad."
    ),
    "cricket_basics": (
        "Cricket is played between two teams of eleven players. "
        "A batter tries to score runs by hitting the ball and running between the wickets. "
        "A Test match can last up to five days. "
        "The T20 format finishes in about three hours and is very popular in India."
    ),
    "solar_system": (
        "The solar system has eight planets orbiting the Sun. "
        "Jupiter is the largest planet and has dozens of moons. "
        "Mercury is the closest planet to the Sun. "
        "Earth is the only planet known to support life."
    ),
    "classical_music": (
        "Beethoven composed nine symphonies during his lifetime. "
        "Mozart began composing music when he was only five years old. "
        "The orchestra is divided into strings, woodwinds, brass, and percussion. "
        "Bach is famous for his intricate fugues and organ works."
    ),
}

In [16]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))


[Source: solar_system]
The solar system has eight planets orbiting the Sun. Jupiter is the largest planet and has dozens of moons. Mercury is the closest planet to the Sun. Earth is the only planet known to support life.

[Source: cricket_basics]
Cricket is played between two teams of eleven players. A batter tries to score runs by hitting the ball and running between the wickets. A Test match can last up to five days. The T20 format finishes in about three hours and is very popular in India.


In [21]:
# @title Ask gemini to search for something
import os
import time
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.8-flash",
    tools=[search_documents],
    system_instruction=(
        "You are a helpful assistant. "
        "When you need facts, use the search_documents tool. "
        "Only answer based on what you find in the documents. "
        "If the documents do not contain the answer, say so clearly."
    ),
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.8-flash


In [18]:
!pip install -q google-generativeai
print("Installed")

Installed


In [24]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=5):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents, config=config
            )
        except Exception as e:
            if ("429" in str(e) or "503" in str(e)) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("Where did pizza originate?")

Question: Where did pizza originate?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'pizza originate origin history'})
  Result preview: [Source: pizza_facts]
Pizza originated in Naples, Italy, in the 18th century. The classic Margherita...

Final answer:
Pizza originated in **Naples, Italy**, in the 18th century.


In [25]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "Where did pizza originate?",
    "How long can a Test cricket match last?",
    "Which planet is the largest in the solar system?",
    "How many symphonies did Beethoven compose?",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: Where did pizza originate?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'Where did pizza originate'})
  Result preview: [Source: pizza_facts]
Pizza originated in Naples, Italy, in the 18th century. The classic Margherita...

Final answer:
Pizza originated in **Naples, Italy**, during the 18th century.

Question: How long can a Test cricket match last?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'Test cricket match duration days hours'})
  Result preview: [Source: cricket_basics]
Cricket is played between two teams of eleven players. A batter tries to sc...

Final answer:
A Test cricket match can last up to **five days**.

Question: Which planet is the largest in the solar system?
--------------------------------------------------
  Rate limit hit, waiting 20s...
  Rate limit hit, waiting 40s...

Final answer:
The largest planet in our solar system is **Jupiter**. It is 

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 2h17m19.246370584s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.g

In [29]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)